In [1]:
# ============================================================================
# EXTRACTION D'EMBEDDINGS VGG16 A PARTIR DE MEL-SPECTROGRAMMES
# VERSION OPTIMISEE MEMOIRE (GOOGLE COLAB)
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Installation des dependances

In [2]:
!pip install -q pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.0 MB/s eta 0:00:00


## 3. Imports

In [3]:
import os
import gc
import zipfile
import tempfile

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torchvision.transforms as T
import torchvision.models as models

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from tqdm import tqdm

import pyarrow as pa
import pyarrow.parquet as pq

## 4. Configuration

In [4]:
PATH_ZIP = "/content/drive/MyDrive/audio_simon_moutier/Data/Audio/Spectrogramme/3s_mel_spectrograms_subset.zip"
PATH_OUTPUT_PARQUET = "/content/drive/MyDrive/audio_simon_moutier/embeddings/vgg16_embeddings_subset.parquet"

BATCH_SIZE = 16
NUM_WORKERS = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

Device : cuda


## 5. Chargement VGG16 et transformations

In [5]:
print("Chargement VGG16...")

vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

features_extractor = vgg16.features
embedding_head = torch.nn.Sequential(*list(vgg16.classifier.children())[:-1])

features_extractor = features_extractor.to(DEVICE)
embedding_head = embedding_head.to(DEVICE)

if DEVICE.type == 'cuda':
    features_extractor = features_extractor.half()
    embedding_head = embedding_head.half()
    print("FP16 active")

features_extractor.eval()
embedding_head.eval()
print("Modele charge")

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Transformations pretes")

Chargement VGG16...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 110MB/s] 


FP16 active
Modele charge
Transformations pretes


## 6. Dataset direct ZIP et DataLoader

In [6]:
class ZipSpectrogramDataset(Dataset):
    def __init__(self, zip_path, transform=None):
        self.zip_path = zip_path
        self.transform = transform
        self.zip_file = zipfile.ZipFile(zip_path)
        self.image_files = sorted([
            f for f in self.zip_file.namelist()
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        print(f"{len(self.image_files)} images trouvees")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        with self.zip_file.open(img_name) as file:
            image = Image.open(file).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, os.path.basename(img_name)


dataset = ZipSpectrogramDataset(PATH_ZIP, transform=transform)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"{len(dataset)} images pretes")

10325 images trouvees
10325 images pretes


## 7. Initialisation Parquet et extraction

In [7]:
embedding_dim = 4096

schema = pa.schema(
    [('filename', pa.string())] + [(f'dim_{i}', pa.float32()) for i in range(embedding_dim)]
)

parquet_writer = pq.ParquetWriter(
    PATH_OUTPUT_PARQUET,
    schema=schema,
    compression='snappy'
)

print("Writer parquet initialise")
print("\nExtraction des embeddings...")

processed = 0

with torch.no_grad():
    for batch_imgs, batch_names in tqdm(dataloader):
        batch_imgs = batch_imgs.to(DEVICE, non_blocking=True)

        if DEVICE.type == 'cuda':
            batch_imgs = batch_imgs.half()

        with torch.amp.autocast(device_type='cuda', enabled=(DEVICE.type == 'cuda')):
            x = features_extractor(batch_imgs)
            x = torch.flatten(x, 1)
            embeddings = embedding_head(x)

        embeddings = embeddings.float().cpu().numpy()

        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        embeddings = embeddings / (norms + 1e-12)

        batch_df = pd.DataFrame(embeddings, columns=[f'dim_{i}' for i in range(embedding_dim)])
        batch_df.insert(0, 'filename', batch_names)

        table = pa.Table.from_pandas(batch_df, schema=schema, preserve_index=False)
        parquet_writer.write_table(table)

        processed += len(batch_names)

        del batch_imgs
        del embeddings
        del batch_df
        del table

        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

parquet_writer.close()
print("\nParquet sauvegarde")

Writer parquet initialise

Extraction des embeddings...


100%|██████████| 646/646 [30:58<00:00,  2.88s/it]



Parquet sauvegarde
